# Stable Matching & Gale-Shapley Algorithm Demo
稳定匹配与 Gale-Shapley 算法演示
## 1. Problem Background / 问题背景
The Stable Matching Problem aims to find a stable one-to-one assignment between two equally sized sets (e.g., $N$ men and $N$ women), based on their strict preferences.\
稳定匹配问题旨在基于严格的偏好列表，在两个等大集合（例如$N$男和$N$女）之间找到一种稳定的一对一分配方案。
## 2. How to DEFINATE "Stable"? / 怎么定义“稳定”？
A matching is unstable if there is a Blocking Pair: a man and a woman who are not matched to each other, but both prefer each other over their current partners. A matching is stable when no such pairs exist.\
如果存在阻塞对 (Blocking Pair)，即一对未匹配的男女，他们都认为对方比自己当前的伴侣更好，则匹配是不稳定的。当不存在任何阻塞对时，匹配就是稳定的。
## 3. The Gale-Shapley Algorithm / 算法流程
The Gale-Shapley (Deferred Acceptance) algorithm guarantees a stable matching in $O(N^2)$ time. The process is as follows:\
Gale-Shapley（延迟接受）算法保证能在$O(N^2)$时间内找到一个稳定匹配。流程如下：\
1. Propose / 求婚： \
Unengaged men propose to their most preferred woman who hasn't rejected them yet.\
单身男性向偏好列表中尚未拒绝过自己的最喜爱女性求婚。
2. Defer & Reject / 延迟接受与拒绝： \
Women temporarily accept the best proposal they've received so far and reject the rest. If a better offer comes later, they dump their current partner to accept the new one.\
女性暂时接受目前收到的最佳求婚，并拒绝其他。如果之后遇到更好的求婚者，则抛弃现任，接受新欢。
3. Repeat / 重复： \
The process repeats until everyone is matched.\
重复此过程，直到所有人成功匹配。
## 4. Proposer-Optimal / 提议方最优特性
A crucial property of this algorithm is that it is proposer-optimal. The proposing side (in this case, men) gets the best possible valid partner among all possible stable matchings, while the receiving side (women) gets the worst.\
该算法的一个关键特性是提议方最优。主动求婚的一方（此例中为男方）将获得所有可能的稳定匹配方案中对其最有利的结果，而被动接受的一方（女方）将获得最差的结果。

# 5. Data Structures & Initialization / 数据结构与初始化
## 5.1 Preference Lists / 偏好列表
We use dictionaries to store the original preference rankings.\
我们使用字典来存储原始的偏好排名。\
- men_prefs: Keys are men's names, values are lists of women ordered by preference. (键是男性名字，值是按偏好降序排列的女性列表)\
- women_prefs: Keys are women's names, values are lists of men ordered by preference. (键是女性名字，值是按偏好降序排列的男性列表)
## 5.2 State Trackers / 状态追踪器
These variables track the ongoing progress of the algorithm.\这些变量用于在算法循环中追踪当前的执行进度。
- free_men: A list acting as a queue of currently unengaged men. \
(一个列表，作为当前单身男性（等待求婚者）的队列)
- engagements: A dictionary mapping a woman to her current fiancé. Storing it as "Woman -> Man" makes it extremely fast to check her current status. \
(一个字典，记录当前的订婚状态。以“女方 -> 男方”的形式存储，方便女方快速查询现任)
- proposals_count: A dictionary tracking the index of the next woman each man should propose to. This prevents us from destroying the original preference lists. \
(一个字典，充当指针，记录每个男性接下来该向名单里的第几名求婚。这避免了修改或破坏原始偏好列表)
## 5.3 The Ranking Index (Optimization) / 排名倒排索引（性能优化）
women_rankings: A nested dictionary storing the exact rank of each man for each woman. \
(一个嵌套字典，本质上是哈希表，预先存储每个女性心中各个男性的具体名次)\
Why we need this / 为什么需要它：\
If a woman $W$ needs to compare her current partner $M_1$ with a new proposer $M_2$, she can simply look up their integer ranks in this dictionary. This reduces the comparison time from $O(N)$ to $O(1)$, ensuring the overall algorithm strictly runs in $O(N^2)$ time. \
如果女性 $W$ 需要比较现任 $M_1$ 和新求婚者 $M_2$，她只需在这个字典中直接读取两人的名次整数。这使得比较操作的时间复杂度从 $O(N)$ 降至 $O(1)$，确保算法整体以严格的 $O(N^2)$ 时间复杂度运行。

In [2]:
####### Hardcoding the Data for Test #######
# 这里我们构造一个经典的 4男 4女 测试用例
men_prefs = {
    "M1": ["W1", "W2", "W3", "W4"],
    "M2": ["W2", "W1", "W4", "W3"],
    "M3": ["W3", "W4", "W1", "W2"],
    "M4": ["W4", "W3", "W2", "W1"]
}

women_prefs = {
    "W1": ["M4", "M3", "M2", "M1"],
    "W2": ["M3", "M4", "M1", "M2"],
    "W3": ["M2", "M1", "M4", "M3"],
    "W4": ["M1", "M2", "M3", "M4"]
}


In [5]:
####### Initializing Trackers #######

### Getting Data ###
proposer_prefs = men_prefs
receiver_prefs = women_prefs
# As setting trackers in 5.2
# 提取所有男性的名字，初始状态全部单身
free_proposers = list(proposer_prefs.keys())  
# 初始没有任何人配对，格式将为 接收方:提议方 (Receiver: Proposer)
engagements = {}  
# 字典推导式，所有提议方的提议次数初始为 0
proposals_count = {p: 0 for p in proposer_prefs}  

### Building the Hash Table ###
# 将接收方的偏好列表转化为 姓名:名次 的 O(1) 查询字典
receiver_rankings = {}
for r, prefs in receiver_prefs.items():
    # 为每一位接收方初始化一个空的内部字典
    receiver_rankings[r] = {}  
    for rank, p in enumerate(prefs):
        # 记录该提议方在当前接收方心中的具体排名（数字越小越靠前）
        receiver_rankings[r][p] = rank

# 打印一下构建好的哈希表，方便演示时查看
# print("构建完成的女性偏好哈希表：")
# for w, ranks in women_rankings.items():
#     print(f"{w}: {ranks}")

In [6]:
####### Algorithm Implementation ############

# 只要提议方队列不为空，循环就继续
while free_proposers:
    # 从队列中取出一个单身的提议方
    p = free_proposers.pop(0)
    
    # 根据该提议方的进度指针，找到下一个接收方对象
    # proposer_prefs 是我们在初始化阶段传入的提议方偏好字典
    r = proposer_prefs[p][proposals_count[p]]
    
    # 无论结果如何，该提议方的进度指针都要加 1，确保下次尝试下一个对象
    proposals_count[p] += 1
    
    # 接收方做出决定
    if r not in engagements:
        # 情况A：接收方目前单身，直接接受提议
        engagements[r] = p
    else:
        # 情况B：接收方已在“订婚”状态，需要进行比较
        p_current = engagements[r]
        
        # 利用之前构建的 receiver_rankings 哈希表进行 O(1) 排名比较
        # 数字越小，代表排名越靠前（越喜欢）
        if receiver_rankings[r][p] < receiver_rankings[r][p_current]:
            # 新的提议方排名更高，接受新欢
            engagements[r] = p
            # 原配被甩，重新加入单身的提议方队列
            free_proposers.append(p_current)
        else:
            # 原配更好，拒绝新的提议
            # 该提议方继续保持单身，回到队列等待下一轮向下一位尝试
            free_proposers.append(p)

# 4. Result Output / 结果输出
print("--- Final Stable Matching Result ---")
# engagements 字典的存储格式为 {接收方: 提议方}
for r, p in engagements.items():
    # 打印时我们统一以“提议方”作为主语，方便阅读
    print(f"{p} is matched with {r}")

--- Final Stable Matching Result ---
M1 is matched with W1
M2 is matched with W2
M3 is matched with W3
M4 is matched with W4
